# Training Analysis

Reads TensorBoard event files from a training run and plots:
- Training losses over steps (total, box, dfl, cls, dist, poly)
- Validation metrics over epochs (mAP, mAP50, AR100, F1score50)
- Learning rate schedule over steps

Plots are saved to `{output_dir}/analysis_plots/`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
output_dir = "/tmp/yolo_run"  # change to your run's output directory

import os
import glob

tb_events_dir = os.path.join(output_dir, "tb_events")
plots_dir = os.path.join(output_dir, "analysis_plots")
os.makedirs(plots_dir, exist_ok=True)

print(f"Reading TensorBoard events from: {tb_events_dir}")
print(f"Saving plots to:                 {plots_dir}")

## Load TensorBoard Scalars

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import collections

def load_scalars(events_dir):
    """Load all scalar tags from a TensorBoard event directory.

    Returns a dict mapping tag -> list of (step, value) tuples, sorted by step.
    Handles multiple event files (e.g. from training restarts) by merging them.
    """
    all_data = collections.defaultdict(dict)  # tag -> {step: value}

    event_files = sorted(glob.glob(os.path.join(events_dir, "**", "events.out.tfevents.*"), recursive=True))
    if not event_files:
        # Also look directly in the dir
        event_files = sorted(glob.glob(os.path.join(events_dir, "events.out.tfevents.*")))

    print(f"Found {len(event_files)} event file(s).")

    for ef in event_files:
        ea = EventAccumulator(ef, size_guidance={"scalars": 0})
        ea.Reload()
        for tag in ea.Tags().get("scalars", []):
            for scalar_event in ea.Scalars(tag):
                all_data[tag][scalar_event.step] = scalar_event.value

    # Convert to sorted lists
    result = {}
    for tag, step_val in all_data.items():
        sorted_pairs = sorted(step_val.items())
        result[tag] = ([s for s, _ in sorted_pairs], [v for _, v in sorted_pairs])

    print(f"Loaded {len(result)} scalar tags: {sorted(result.keys())}")
    return result


scalars = load_scalars(tb_events_dir)

## Training Losses

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe backend
import matplotlib.pyplot as plt

LOSS_TAGS = [
    ("train/total_loss", "Total loss"),
    ("train/box_loss",   "Box (CIoU)"),
    ("train/dfl_loss",   "DFL"),
    ("train/cls_loss",   "Class"),
    ("train/dist_loss",  "Distance"),
    ("train/poly_loss",  "Polygon"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (tag, label) in zip(axes, LOSS_TAGS):
    if tag in scalars:
        steps, values = scalars[tag]
        ax.plot(steps, values, linewidth=0.8)
        ax.set_title(label)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{label} (not found)")
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

fig.suptitle("Training Losses", fontsize=14)
fig.tight_layout()
save_path = os.path.join(plots_dir, "training_losses.png")
fig.savefig(save_path, dpi=120)
plt.close(fig)
print(f"Saved: {save_path}")

## Validation Metrics

In [ ]:
VAL_TAGS = [
    ("val/mAP",       "mAP (0.5:0.95)"),
    ("val/mAP50",     "mAP50"),
    ("val/AR100",     "AR100"),
    ("val/F1score50", "F1@50"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (tag, label) in zip(axes, VAL_TAGS):
    if tag in scalars:
        steps, values = scalars[tag]
        ax.plot(steps, values, marker="o", markersize=3)
        ax.set_title(label)
        ax.set_xlabel("Step")
        ax.set_ylabel("Value")
        ax.grid(True, alpha=0.3)
        best_val = max(values)
        best_step = steps[values.index(best_val)]
        ax.annotate(f"best={best_val:.4f}\n@step {best_step}",
                    xy=(best_step, best_val), fontsize=8,
                    xytext=(0.05, 0.85), textcoords="axes fraction")
    else:
        ax.set_title(f"{label} (not found)")
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)

fig.suptitle("Validation Metrics", fontsize=14)
fig.tight_layout()
save_path = os.path.join(plots_dir, "val_metrics.png")
fig.savefig(save_path, dpi=120)
plt.close(fig)
print(f"Saved: {save_path}")

## Learning Rate Schedule

In [ ]:
LR_TAG = "train/learning_rate"

fig, ax = plt.subplots(figsize=(10, 4))

if LR_TAG in scalars:
    steps, values = scalars[LR_TAG]
    ax.plot(steps, values, linewidth=0.8, color="tab:orange")
    ax.set_title("Learning Rate")
    ax.set_xlabel("Step")
    ax.set_ylabel("LR")
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, f"Tag '{LR_TAG}' not found", ha="center", va="center",
            transform=ax.transAxes)
    print(f"Available tags: {sorted(scalars.keys())}")

fig.tight_layout()
save_path = os.path.join(plots_dir, "learning_rate.png")
fig.savefig(save_path, dpi=120)
plt.close(fig)
print(f"Saved: {save_path}")
print(f"\nAll plots written to: {plots_dir}")